In [ ]:
# Install and import
!pip install ultralytics opencv-python-headless pandas



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 63.4M/63.4M [00:00<00:00, 134MB/s]


✅ Done: Video saved to 'annotated_output.mp4'
📄 CSV saved to 'detections.csv'
🧾 JSON saved to 'detections.json'


In [ ]:
import cv2
import json
import pandas as pd
from ultralytics import RTDETR

# --- Paths ---
VIDEO_PATH = "/content/2103099-uhd_3840_2160_30fps.mp4"
OUTPUT_VIDEO_PATH = "annotated_output.mp4"
CSV_PATH = "detections.csv"
JSON_PATH = "detections.json"

# --- Load Model ---
model = RTDETR("rtdetr-l.pt")

# --- Prepare Video Reader/Writer ---
cap = cv2.VideoCapture(VIDEO_PATH)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

# --- Setup Logging ---
csv_data = []
json_data = []
frame_count = 0

try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1

        results = model(frame)
        annotated_frame = results[0].plot()

        names = results[0].names
        boxes = results[0].boxes
        counts = {}
        for box in boxes:
            cls_id = int(box.cls)
            label = names[cls_id]
            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            counts[label] = counts.get(label, 0) + 1

            csv_data.append([frame_count, label, round(conf, 2), x1, y1, x2, y2])
            json_data.append({
                "frame": frame_count,
                "label": label,
                "confidence": conf,
                "bbox": [x1, y1, x2, y2]
            })

        # Count text
        y_pos = 20
        for label, count in counts.items():
            cv2.putText(annotated_frame, f"{label}: {count}", (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            y_pos += 20

        out.write(annotated_frame)

finally:
    cap.release()
    out.release()

    df = pd.DataFrame(csv_data, columns=["Frame", "Label", "Confidence", "X1", "Y1", "X2", "Y2"])
    df.to_csv(CSV_PATH, index=False)

    with open(JSON_PATH, "w") as f:
        json.dump(json_data, f, indent=2)

    print(f"✅ Done: Video saved to '{OUTPUT_VIDEO_PATH}'")
    print(f"📄 CSV saved to '{CSV_PATH}'")
    print(f"🧾 JSON saved to '{JSON_PATH}'")



0: 640x640 4 persons, 20 cars, 1 bus, 14 trucks, 3 traffic lights, 86.6ms
Speed: 6.0ms preprocess, 86.6ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 persons, 21 cars, 2 buss, 13 trucks, 3 traffic lights, 115.9ms
Speed: 6.0ms preprocess, 115.9ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 4 persons, 22 cars, 2 buss, 13 trucks, 2 traffic lights, 119.6ms
Speed: 5.8ms preprocess, 119.6ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 5 persons, 24 cars, 1 bus, 13 trucks, 3 traffic lights, 126.3ms
Speed: 5.8ms preprocess, 126.3ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 5 persons, 21 cars, 2 buss, 15 trucks, 2 traffic lights, 72.2ms
Speed: 6.9ms preprocess, 72.2ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 6 persons, 22 cars, 1 bus, 12 trucks, 2 traffic lights, 70.6ms
Speed: 4.5ms preprocess, 70.6ms inference, 0.8ms postpro

KeyboardInterrupt: 